# Inference: Significance on Train vs. Holdout

We reuse the course methods for inference on our binary outcome using logistic regression. The notebook assumes an "optimal model" selected during model selection; when available, set `OPTIMAL_MODEL_PATH` to load it. Otherwise, we refit a statsmodels Logit on the encoded features to obtain coefficient estimates and standard errors.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
np.random.seed(42)

OPTIMAL_MODEL_PATH = Path("results/optimal_model.pkl")


In [ ]:

# Load preprocessed training data and create encoded feature matrix
def load_encoded_train(train_path="data/train_data_preprocessed.csv"):
    df = pd.read_csv(train_path)
    X = df.drop(columns=["y"])
    y = df["y"].map({"yes": 1, "no": 0})
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    X_enc = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    return X_enc, y

# Encode holdout with train columns to keep alignment
# If you already have a preprocessed holdout file, point holdout_path there instead.
def load_encoded_holdout(train_cols, holdout_path="data/holdout_data.csv"):
    holdout_df = pd.read_csv(holdout_path)
    if "y" in holdout_df.columns:
        y_holdout = holdout_df["y"].map({"yes": 1, "no": 0})
        X_holdout = holdout_df.drop(columns=["y"])
    else:
        y_holdout = None
        X_holdout = holdout_df
    cat_cols = [c for c in X_holdout.columns if X_holdout[c].dtype == "object"]
    X_holdout_enc = pd.get_dummies(X_holdout, columns=cat_cols, drop_first=True)
    # align to train columns
    X_holdout_enc = X_holdout_enc.reindex(columns=train_cols, fill_value=0)
    return X_holdout_enc, y_holdout

train_X, train_y = load_encoded_train()
holdout_X, holdout_y = load_encoded_holdout(train_X.columns)
print(f"Train encoded shape: {train_X.shape}; Holdout encoded shape: {holdout_X.shape}")


## (a) Significant coefficients on training data
Fit statsmodels Logit on the encoded training set, compute p-values, and flag coefficients below our significance threshold (alpha = 0.05 by default).

In [ ]:

alpha = 0.05

# Optionally load an existing optimal model for predictions; for inference we fit statsmodels for proper SEs
X_const = sm.add_constant(train_X)
logit_train = sm.Logit(train_y, X_const)
train_res = logit_train.fit(disp=False)

summary_table = train_res.summary2().tables[1]
summary_table['significant'] = summary_table['P>|z|'] < alpha
significant = summary_table[summary_table['significant']]

print(f"Significant at alpha={alpha}: {len(significant)} / {len(summary_table)}")
display(summary_table.sort_values('P>|z|').head(15))


### Interpretation
Use this section to describe in words what statistical significance means for the flagged coefficients and why you chose alpha = 0.05. Focus on the most practically meaningful variables rather than exhaustively listing all significant coefficients.

## (b) Significance on held-out data
Refit the same specification on the holdout set and compare which coefficients stay significant.

In [ ]:

# Only run if holdout has labels
auto_run = holdout_y is not None and not holdout_y.isnull().all()
if not auto_run:
    print("Holdout lacks labels; add a 'y' column with yes/no to run this section.")
else:
    holdout_const = sm.add_constant(holdout_X)
    logit_holdout = sm.Logit(holdout_y, holdout_const)
    holdout_res = logit_holdout.fit(disp=False)
    holdout_summary = holdout_res.summary2().tables[1]
    holdout_summary['significant'] = holdout_summary['P>|z|'] < alpha
    significant_holdout = holdout_summary[holdout_summary['significant']]

    print(f"Holdout significant at alpha={alpha}: {len(significant_holdout)} / {len(holdout_summary)}")
    display(holdout_summary.sort_values('P>|z|').head(15))


### Compare train vs. holdout
Summarize which coefficients remain significant, which drop out, and plausible reasons (sampling variability, class imbalance, different mix of customers, etc.).